### Phase 1 — Data Quality & Preparation


#### Load Libraries & Raw Data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

products    = pd.read_csv('data/olist_products_dataset.csv')
order_items = pd.read_csv('data/olist_order_items_dataset.csv')
orders      = pd.read_csv('data/olist_orders_dataset.csv')
customers   = pd.read_csv('data/olist_customers_dataset.csv')
reviews     = pd.read_csv('data/olist_order_reviews_dataset.csv')
payments    = pd.read_csv('data/olist_order_payments_dataset.csv')

print("  RAW DATASETS")
print("  " + "=" * 44)
print(f"  products      : {products.shape}")
print(f"  order_items   : {order_items.shape}")
print(f"  orders        : {orders.shape}")
print(f"  customers     : {customers.shape}")
print(f"  reviews       : {reviews.shape}")
print(f"  payments      : {payments.shape}")


  RAW DATASETS  (Full Olist)
  products      : (32951, 9)
  order_items   : (112650, 7)
  orders        : (99441, 8)
  customers     : (99441, 5)
  reviews       : (99224, 7)
  payments      : (103886, 5)


#### Scope Data to Housewares

In [ ]:
hw_products  = products[products['product_category_name'] == 'utilidades_domesticas'].copy()
hw_items     = order_items[order_items['product_id'].isin(hw_products['product_id'])].copy()
hw_order_ids = hw_items['order_id'].unique()
hw_orders    = orders[orders['order_id'].isin(hw_order_ids)].copy()
hw_customers = customers[customers['customer_id'].isin(hw_orders['customer_id'])].copy()
hw_reviews   = reviews[reviews['order_id'].isin(hw_order_ids)].copy()
hw_payments  = payments[payments['order_id'].isin(hw_order_ids)].copy()

print("\n  HOUSEWARES SCOPE")
print("  " + "=" * 44)
print(f"  Distinct Products : {len(hw_products):,}")
print(f"  Order Item Rows   : {len(hw_items):,}")
print(f"  Unique Orders     : {len(hw_orders):,}")
print(f"  Customers         : {len(hw_customers):,}")
print(f"  Reviews           : {len(hw_reviews):,}")
print(f"  Payment Rows      : {len(hw_payments):,}")



  HOUSEWARES SCOPE  (utilidades_domesticas)
  Distinct Products : 2,335
  Order Item Rows   : 6,964
  Unique Orders     : 5,884
  Customers         : 5,884
  Reviews           : 5,865
  Payment Rows      : 6,240


#### Data Quality Checks
##### Issue 1 — Missing Delivery Timestamps
Fields: `order_delivered_customer_date` and `order_delivered_carrier_date`


In [54]:
missing_orders = hw_orders[[
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]].isnull().sum().rename('Null Count')

print("\n  Missing Delivery Timestamps")
print("  " + "=" * 44)
for col, val in missing_orders.items():
    print(f"  {col:<30} : {val:>5}")

print("\n  Orders breakdown by status:")
print("  " + "=" * 44)
for status, val in hw_orders['order_status'].value_counts().items():
    print(f"  {status:<30} : {val:>5,}")



  Missing Delivery Timestamps
  order_status                   :     0
  order_purchase_timestamp       :     0
  order_approved_at              :     0
  order_delivered_carrier_date   :    61
  order_delivered_customer_date  :   141
  order_estimated_delivery_date  :     0

  Orders breakdown by status:
  delivered                      : 5,743
  shipped                        :    76
  canceled                       :    37
  processing                     :    17
  invoiced                       :    10
  approved                       :     1


##### Issue 2 — Chronological Timestamp Inversions

In [55]:
for col in ['order_purchase_timestamp', 'order_approved_at',
            'order_delivered_carrier_date', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    hw_orders[col] = pd.to_datetime(hw_orders[col])

delivered_only  = hw_orders[hw_orders['order_status'] == 'delivered'].copy()
invalid_carrier = (delivered_only['order_purchase_timestamp'] > delivered_only['order_delivered_carrier_date']).sum()
invalid_deliver = (delivered_only['order_delivered_carrier_date'] > delivered_only['order_delivered_customer_date']).sum()
late_count      = (delivered_only['order_delivered_customer_date'] > delivered_only['order_estimated_delivery_date']).sum()
total_delivered = len(delivered_only)

print("\n  Chronological Timestamp Inversions")
print("  " + "=" * 48)
print(f"  {'Anomaly: Purchase AFTER carrier handoff':<42} : {invalid_carrier} orders")
print(f"  {'Anomaly: Carrier handoff AFTER delivery':<42} : {invalid_deliver} orders")
print(f"  {'Late deliveries (actual > estimated)':<42} : {late_count} out of {total_delivered:,}")
print(f"  {'On-time delivery rate (delivered orders)':<42} : {round((1 - late_count/total_delivered)*100, 2)}%")



  Chronological Timestamp Inversions
  Anomaly: Purchase AFTER carrier handoff    : 9 orders
  Anomaly: Carrier handoff AFTER delivery    : 3 orders
  Late deliveries (actual > estimated)       : 399 out of 5,743
  On-time delivery rate (delivered orders)   : 93.05%


##### Issue 3 — Missing Review Comment Text (`review_comment_message`)

In [56]:
missing_reviews = hw_reviews[['review_score', 'review_comment_message']].isnull().sum().rename('Null Count')
print("\n  Missing Values in Reviews Table")
print("  " + "=" * 44)
for col, val in missing_reviews.items():
    print(f"  {col:<30} : {val:>5,}")
print(f"\n  {'Missing written comment rate':<30} : {hw_reviews['review_comment_message'].isnull().mean()*100:.1f}%")
print("  -> Resolution: review_score (0 nulls) is used for satisfaction analysis instead.")


  Missing Values in Reviews Table
  review_score                   :     0
  review_comment_message         : 3,497

  Missing written comment rate   : 59.6%
  -> Resolution: review_score (0 nulls) is used for satisfaction analysis instead.


##### Issue 4 — Orders With No Review (Orphan Orders)

In [57]:
orders_without_review = set(hw_order_ids) - set(hw_reviews['order_id'])
print("\n  Orders With No Review (Orphan Orders)")
print("  " + "=" * 44)
print(f"  {'Houseware orders with no review entry':<38} : {len(orders_without_review)}")
print(f"  {'As % of total Houseware orders':<38} : {round(len(orders_without_review)/len(hw_order_ids)*100, 2)}%")
print("\n  -> Resolution: LEFT JOIN for operations; INNER JOIN when correlating delay with rating.")



  Orders With No Review (Orphan Orders)
  Houseware orders with no review entry  : 41
  As % of total Houseware orders         : 0.7%

  -> Resolution: LEFT JOIN for operations; INNER JOIN when correlating delay with rating.


##### Issue 5 — Dual Customer Identifier (Repeat Buyers Masked)

In [58]:
dup_cust_id  = hw_customers['customer_id'].duplicated().sum()
repeat_uid   = hw_customers['customer_unique_id'].duplicated().sum()
total_unique = hw_customers['customer_unique_id'].nunique()

print("\n  Dual Customer Identifier (Repeat Buyers)")
print("  " + "=" * 44)
print(f"  {'Duplicate customer_id (per-order key)':<38} : {dup_cust_id}")
print(f"  {'Repeat buyers via customer_unique_id':<38} : {repeat_uid}")
print(f"  {'Truly distinct Houseware customers':<38} : {total_unique:,}")
print("\n  -> Resolution: Group by customer_unique_id for any retention analysis.")



  Dual Customer Identifier (Repeat Buyers)
  Duplicate customer_id (per-order key)  : 0
  Repeat buyers via customer_unique_id   : 63
  Truly distinct Houseware customers     : 5,821

  -> Resolution: Group by customer_unique_id for any retention analysis.
